In [ ]:
!pip install -q crewai groq gradio

In [ ]:
import time
import gradio as gr
from groq import Groq
from crewai import Agent, Task, Crew

In [ ]:
GROQ_API_KEY = "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"

groq_client = Groq(api_key=GROQ_API_KEY)
MODEL_NAME = "llama-3.1-8b-instant"

def call_llm(prompt):
    response = groq_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

In [ ]:
SCALE = 1000  # Fixed-point scaling factor

def to_fixed(x):
    return int(x * SCALE)

def from_fixed(x):
    return x / SCALE

In [ ]:
def state_collector(current_value, target_value):
    return {
        "current": to_fixed(current_value),
        "target": to_fixed(target_value)
    }

In [ ]:
def goal_evaluator(state):
    error = abs(state["target"] - state["current"])
    state["error"] = error
    return state

In [ ]:
def action_simulator(state):

    increase_state = state["current"] + to_fixed(0.5)
    decrease_state = state["current"] - to_fixed(0.5)

    inc_error = abs(state["target"] - increase_state)
    dec_error = abs(state["target"] - decrease_state)

    state["inc_error"] = inc_error
    state["dec_error"] = dec_error

    return state

In [ ]:
def decision_selector(state):

    if state["inc_error"] < state["dec_error"]:
        state["decision"] = "INCREASE_CONTROL_SIGNAL"
        state["new_value"] = state["current"] + to_fixed(0.5)
    else:
        state["decision"] = "DECREASE_CONTROL_SIGNAL"
        state["new_value"] = state["current"] - to_fixed(0.5)

    return state

In [ ]:
def command_output(state):
    return {
        "command": state["decision"],
        "updated_value": from_fixed(state["new_value"]),
        "error": from_fixed(state["error"])
    }

In [ ]:
def rtos_frame_execution(current_value, target_value, frame_ms=20):

    start_time = time.perf_counter()

    state = state_collector(current_value, target_value)
    state = goal_evaluator(state)
    state = action_simulator(state)
    state = decision_selector(state)
    output = command_output(state)

    wcet = (time.perf_counter() - start_time) * 1000  # ms

    frame_status = "FRAME OK" if wcet < frame_ms else "WCET EXCEEDED"

    output["wcet_ms"] = wcet
    output["frame_status"] = frame_status

    return output

In [ ]:
report_agent = Agent(
    role="RTOS Aerospace Reporting Officer",
    goal="Generate professional ARINC 653 partition execution report",
    backstory="You are a senior avionics certification engineer.",
    verbose=False
)

In [ ]:
def generate_report(output):

    prompt = f"""
    Generate a professional aerospace certification-style report.

    Command Issued: {output['command']}
    Updated Control Value: {output['updated_value']}
    Error Remaining: {output['error']}
    WCET (ms): {output['wcet_ms']}
    Frame Status: {output['frame_status']}

    Include:
    - Partition Health
    - Determinism Status
    - Safety Assessment
    - RTOS Compliance Note
    """

    return call_llm(prompt)

In [ ]:
custom_css = """
body {
    background-color: #f4f7fb;
}
.banner {
    white-space: nowrap;
    overflow: hidden;
}
.banner span {
    display: inline-block;
    padding-left: 100%;
    animation: marquee 15s linear infinite;
    font-weight: bold;
    color: #0b3d91;
}
@keyframes marquee {
    0% { transform: translateX(0); }
    100% { transform: translateX(-100%); }
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:

    gr.HTML("""
    <div class="banner">
        <span>RTOS-Based Agent Architecture by Collins Aerospace</span>
    </div>
    """)

    gr.Markdown("## ARINC 653 Goal-Based Agent Partition Simulator")

    with gr.Row():
        current_input = gr.Number(label="Current Value", value=5.0)
        target_input = gr.Number(label="Target Value", value=10.0)
        frame_input = gr.Dropdown([20, 50], value=20, label="Frame Duration (ms)")
        run_button = gr.Button("Execute RTOS Frame", variant="primary")

    output_box = gr.Markdown()

    def run_system(current, target, frame):
        result = rtos_frame_execution(current, target, frame)
        report = generate_report(result)
        return report

    run_button.click(
        run_system,
        inputs=[current_input, target_input, frame_input],
        outputs=output_box
    )

demo.launch(debug=True)